# 🏭 Conveyor Belt Damage Detection

**Complete Training & Inference Pipeline**

This notebook trains a YOLOv8 segmentation model for belt ROI detection, then runs the full damage detection pipeline (scratch + edge damage) on your images.

### Setup Instructions

**For Kaggle:**
1. Upload your dataset as a Kaggle Dataset (zip your `train` folder and upload)
2. Add the dataset to this notebook via "Add Data" → Your Datasets
3. Enable GPU: Settings → Accelerator → GPU P100
4. Run all cells

**For Google Colab:**
1. Upload your `train` folder to Google Drive
2. Enable GPU: Runtime → Change runtime type → GPU (T4)
3. Run all cells (it will mount your Drive automatically)

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q ultralytics opencv-python-headless scipy pyyaml

import os
import sys
import platform

# Detect environment
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IN_COLAB:
    print("✅ Running on Google Colab")
elif IN_KAGGLE:
    print("✅ Running on Kaggle")
else:
    print("⚠️ Running locally")

# Check GPU
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    DEVICE = '0'
else:
    print("⚠️ No GPU found — training will be very slow!")
    DEVICE = 'cpu'

## 2. Mount Data / Set Paths

In [ ]:
# ============================================================
# ⚙️ CONFIGURE YOUR PATHS HERE
# ============================================================

if IN_COLAB:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    # ❗ UPDATE THIS PATH to where your dataset is in Google Drive
    # The folder should contain: train/train/images/ and train/train/labels/
    DRIVE_DATA_PATH = '/content/drive/MyDrive/conveyor_belt_data'
    
    # Copy to local for faster I/O
    SOURCE_DIR = '/content/data/train/train'
    if not os.path.exists(SOURCE_DIR):
        print("📦 Copying dataset from Drive (this may take a minute)...")
        !cp -r "{DRIVE_DATA_PATH}"/* /content/data/
        print("✅ Dataset copied!")
    else:
        print("✅ Dataset already exists locally")

elif IN_KAGGLE:
    # ❗ UPDATE THIS if your Kaggle dataset has a different name
    # Check /kaggle/input/ for the actual dataset directory name
    !ls /kaggle/input/
    
    KAGGLE_DATASET_NAME = os.listdir('/kaggle/input/')[0]  # auto-detect first dataset
    KAGGLE_INPUT = f'/kaggle/input/{KAGGLE_DATASET_NAME}'
    
    # Copy to writable location
    SOURCE_DIR = '/kaggle/working/data/train/train'
    if not os.path.exists(SOURCE_DIR):
        print(f"📦 Copying dataset from {KAGGLE_INPUT}...")
        !cp -r "{KAGGLE_INPUT}"/* /kaggle/working/data/
        print("✅ Dataset copied!")
    else:
        print("✅ Dataset already exists")

else:
    SOURCE_DIR = 'train/train'

# Set working directory
WORK_DIR = '/kaggle/working' if IN_KAGGLE else '/content' if IN_COLAB else '.'
DATASET_DIR = os.path.join(WORK_DIR, 'dataset')
WEIGHTS_DIR = os.path.join(WORK_DIR, 'model_weights')
OUTPUT_DIR = os.path.join(WORK_DIR, 'outputs')

print(f"\n📁 Source dir: {SOURCE_DIR}")
print(f"📁 Work dir:   {WORK_DIR}")
print(f"📁 Dataset:    {DATASET_DIR}")
print(f"📁 Weights:    {WEIGHTS_DIR}")
print(f"📁 Output:     {OUTPUT_DIR}")

In [ ]:
# Verify dataset structure
from pathlib import Path

src = Path(SOURCE_DIR)
img_dir = src / 'train' / 'images'
lbl_dir = src / 'train' / 'labels'

assert img_dir.exists(), f"❌ Images not found at {img_dir}. Check your SOURCE_DIR path!"
assert lbl_dir.exists(), f"❌ Labels not found at {lbl_dir}. Check your SOURCE_DIR path!"

n_imgs = len(list(img_dir.glob('*')))
n_lbls = len(list(lbl_dir.glob('*.txt')))

print(f"✅ Found {n_imgs} images and {n_lbls} labels")
print(f"   Images: {img_dir}")
print(f"   Labels: {lbl_dir}")

## 3. Prepare Dataset (Train/Val Split)

In [ ]:
import random
import shutil
import yaml
from pathlib import Path

def prepare_dataset(source_dir, output_dir, val_split=0.2, seed=42):
    """Reorganize dataset into YOLOv8 train/val structure."""
    source_path = Path(source_dir)
    output_path = Path(output_dir)
    
    images_dir = source_path / 'train' / 'images'
    labels_dir = source_path / 'train' / 'labels'
    
    if not images_dir.exists():
        raise FileNotFoundError(f"Images directory not found: {images_dir}")
    if not labels_dir.exists():
        raise FileNotFoundError(f"Labels directory not found: {labels_dir}")
    
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    image_files = sorted([f for f in images_dir.iterdir() if f.suffix.lower() in image_extensions])
    print(f"Found {len(image_files)} images")
    
    paired_files = []
    for img_file in image_files:
        label_file = labels_dir / (img_file.stem + '.txt')
        if label_file.exists():
            paired_files.append((img_file, label_file))
    
    print(f"Found {len(paired_files)} image-label pairs")
    
    random.seed(seed)
    random.shuffle(paired_files)
    
    val_count = int(len(paired_files) * val_split)
    train_count = len(paired_files) - val_count
    
    train_pairs = paired_files[:train_count]
    val_pairs = paired_files[train_count:]
    
    print(f"Split: {train_count} train, {val_count} val")
    
    for split in ['train', 'val']:
        (output_path / split / 'images').mkdir(parents=True, exist_ok=True)
        (output_path / split / 'labels').mkdir(parents=True, exist_ok=True)
    
    def copy_pairs(pairs, split_name):
        for img_file, label_file in pairs:
            shutil.copy2(img_file, output_path / split_name / 'images' / img_file.name)
            shutil.copy2(label_file, output_path / split_name / 'labels' / label_file.name)
    
    print("Copying training files...")
    copy_pairs(train_pairs, 'train')
    print("Copying validation files...")
    copy_pairs(val_pairs, 'val')
    
    data_yaml = {
        'path': str(output_path.resolve()),
        'train': 'train/images',
        'val': 'val/images',
        'nc': 1,
        'names': ['belt_roi']
    }
    
    yaml_path = output_path / 'data.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump(data_yaml, f, default_flow_style=False)
    
    print(f"\n✅ Dataset prepared: {output_path.resolve()}")
    return str(yaml_path.resolve())


# Prepare the dataset
data_yaml_path = os.path.join(DATASET_DIR, 'data.yaml')
if not os.path.exists(data_yaml_path):
    data_yaml_path = prepare_dataset(SOURCE_DIR, DATASET_DIR, val_split=0.2)
else:
    print(f"✅ Dataset already prepared at {data_yaml_path}")

print(f"\n📄 data.yaml path: {data_yaml_path}")

## 4. Train YOLOv8 Segmentation Model 🚀

In [ ]:
# ============================================================
# ⚙️ TRAINING HYPERPARAMETERS — tweak as needed
# ============================================================

EPOCHS = 100        # Number of training epochs (100 is a good default)
BATCH_SIZE = 8      # Batch size (increase if GPU has enough VRAM)
IMG_SIZE = 640      # Image size for training
MODEL_BASE = 'yolov8n-seg.pt'  # Base model (n=nano, s=small, m=medium)

print(f"Training config:")
print(f"  Model:      {MODEL_BASE}")
print(f"  Epochs:     {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image size: {IMG_SIZE}")
print(f"  Device:     {DEVICE}")

In [ ]:
from ultralytics import YOLO

# Load base model
model = YOLO(MODEL_BASE)

# Train!
results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    device=DEVICE if DEVICE != 'cpu' else DEVICE,
    project=os.path.join(WORK_DIR, 'runs/segment'),
    name='belt_seg',
    exist_ok=True,
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.3,
    flipud=0.5,
    fliplr=0.5,
    mosaic=0.5,
    # Training params
    patience=20,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
)

print("\n✅ Training complete!")

In [ ]:
# Save best weights to a convenient location
import shutil

os.makedirs(WEIGHTS_DIR, exist_ok=True)

best_weights = os.path.join(WORK_DIR, 'runs/segment/belt_seg/weights/best.pt')
last_weights = os.path.join(WORK_DIR, 'runs/segment/belt_seg/weights/last.pt')

BEST_MODEL_PATH = os.path.join(WEIGHTS_DIR, 'belt_seg_best.pt')

if os.path.exists(best_weights):
    shutil.copy2(best_weights, BEST_MODEL_PATH)
    print(f"✅ Best weights saved to: {BEST_MODEL_PATH}")
    print(f"   Size: {os.path.getsize(BEST_MODEL_PATH) / 1e6:.1f} MB")
else:
    print(f"⚠️ best.pt not found, using last.pt")
    if os.path.exists(last_weights):
        shutil.copy2(last_weights, BEST_MODEL_PATH)
        print(f"✅ Last weights saved to: {BEST_MODEL_PATH}")

# Also save to Drive (Colab) so you don't lose it
if IN_COLAB:
    drive_weights = '/content/drive/MyDrive/conveyor_belt_weights/'
    os.makedirs(drive_weights, exist_ok=True)
    shutil.copy2(BEST_MODEL_PATH, drive_weights)
    print(f"✅ Weights also saved to Google Drive: {drive_weights}")

## 5. View Training Results

In [ ]:
# Display training metrics
if hasattr(results, 'results_dict'):
    print("📊 Validation Metrics:")
    print("=" * 40)
    for key, val in results.results_dict.items():
        if isinstance(val, float):
            print(f"  {key}: {val:.4f}")
    print("=" * 40)

In [ ]:
# Display training plots
from IPython.display import Image, display

results_dir = os.path.join(WORK_DIR, 'runs/segment/belt_seg')

for plot_name in ['results.png', 'confusion_matrix.png', 'labels.jpg',
                  'train_batch0.jpg', 'val_batch0_pred.jpg']:
    plot_path = os.path.join(results_dir, plot_name)
    if os.path.exists(plot_path):
        print(f"\n📈 {plot_name}:")
        display(Image(filename=plot_path, width=800))

## 6. Damage Detection Module

The core damage_detector module — defines scratch and edge damage detection using CV techniques within the belt ROI.

In [ ]:
import cv2
import numpy as np
from scipy.ndimage import uniform_filter1d
from typing import List, Tuple, Dict, Optional


class Detection:
    """Single damage detection result."""
    def __init__(self, bbox: Tuple[int, int, int, int], damage_type: str, confidence: float = 1.0):
        self.bbox = bbox
        self.damage_type = damage_type
        self.confidence = confidence

    def __repr__(self):
        return f"Detection({self.damage_type}, bbox={self.bbox}, conf={self.confidence:.2f})"


class ScratchDetector:
    """Detects surface scratches on the conveyor belt via texture anomaly analysis."""
    
    def __init__(self, clahe_clip=3.0, clahe_grid=8, blur_ksize=31, thresh_factor=2.5,
                 min_area=500, min_aspect_ratio=3.0, morph_ksize=5, dilate_iter=2):
        self.clahe_clip = clahe_clip
        self.clahe_grid = clahe_grid
        self.blur_ksize = blur_ksize
        self.thresh_factor = thresh_factor
        self.min_area = min_area
        self.min_aspect_ratio = min_aspect_ratio
        self.morph_ksize = morph_ksize
        self.dilate_iter = dilate_iter

    def detect(self, image, belt_mask):
        h, w = image.shape[:2]
        scale = (h * w) / (1920 * 1080)
        min_area = int(self.min_area * scale)
        
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=self.clahe_clip, tileGridSize=(self.clahe_grid, self.clahe_grid))
        enhanced = clahe.apply(gray)
        masked = cv2.bitwise_and(enhanced, enhanced, mask=belt_mask)
        smoothed = cv2.GaussianBlur(masked, (self.blur_ksize, self.blur_ksize), 0)
        diff = cv2.absdiff(masked, smoothed)
        diff = cv2.bitwise_and(diff, diff, mask=belt_mask)
        
        margin = int(min(h, w) * 0.03)
        kernel_erode = np.ones((margin, margin), np.uint8)
        inner_mask = cv2.erode(belt_mask, kernel_erode, iterations=1)
        diff = cv2.bitwise_and(diff, diff, mask=inner_mask)
        
        belt_pixels = diff[inner_mask > 0]
        if len(belt_pixels) == 0:
            return []
        
        mean_val = np.mean(belt_pixels)
        std_val = np.std(belt_pixels)
        threshold = max(mean_val + self.thresh_factor * std_val, 15)
        
        _, binary = cv2.threshold(diff, threshold, 255, cv2.THRESH_BINARY)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (self.morph_ksize, self.morph_ksize))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        binary = cv2.dilate(binary, kernel, iterations=self.dilate_iter)
        
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        detections = []
        for contour in contours:
            area = cv2.contourArea(contour)
            if area < min_area:
                continue
            x, y, bw, bh = cv2.boundingRect(contour)
            aspect_ratio = max(bw, bh) / (min(bw, bh) + 1e-6)
            if aspect_ratio >= self.min_aspect_ratio:
                roi_diff = diff[y:y+bh, x:x+bw]
                roi_mask = belt_mask[y:y+bh, x:x+bw]
                if np.sum(roi_mask > 0) > 0:
                    intensity = np.mean(roi_diff[roi_mask > 0])
                    conf = min(1.0, intensity / (threshold * 2))
                else:
                    conf = 0.5
                detections.append(Detection((x, y, x + bw, y + bh), 'scratch', conf))
        
        return self._nms(detections, iou_thresh=0.3)
    
    def _nms(self, detections, iou_thresh=0.3):
        if not detections:
            return []
        detections = sorted(detections, key=lambda d: d.confidence, reverse=True)
        keep, suppressed = [], set()
        for i, det_i in enumerate(detections):
            if i in suppressed:
                continue
            keep.append(det_i)
            for j in range(i + 1, len(detections)):
                if j not in suppressed and self._iou(det_i.bbox, detections[j].bbox) > iou_thresh:
                    suppressed.add(j)
        return keep
    
    @staticmethod
    def _iou(b1, b2):
        x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
        x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
        inter = max(0, x2-x1) * max(0, y2-y1)
        a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
        a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
        return inter / (a1 + a2 - inter + 1e-6)


class EdgeDamageDetector:
    """Detects edge damage on the conveyor belt via contour irregularity analysis."""
    
    def __init__(self, smooth_window=51, deviation_threshold=3.0, min_damage_length=20, edge_bbox_width=80):
        self.smooth_window = smooth_window
        self.deviation_threshold = deviation_threshold
        self.min_damage_length = min_damage_length
        self.edge_bbox_width = edge_bbox_width
    
    def detect(self, image, belt_mask):
        h, w = image.shape[:2]
        scale = ((h / 1080) + (w / 1920)) / 2
        smooth_window = max(11, int(self.smooth_window * scale) | 1)
        min_damage_length = max(5, int(self.min_damage_length * scale))
        edge_bbox_width = max(20, int(self.edge_bbox_width * scale))
        
        detections = []
        left_edge, right_edge = self._extract_edges(belt_mask)
        
        if left_edge is not None and len(left_edge) > smooth_window:
            detections.extend(self._detect_anomalies(left_edge, 'left', h, w, smooth_window, min_damage_length, edge_bbox_width))
        if right_edge is not None and len(right_edge) > smooth_window:
            detections.extend(self._detect_anomalies(right_edge, 'right', h, w, smooth_window, min_damage_length, edge_bbox_width))
        
        return detections
    
    def _extract_edges(self, mask):
        h, w = mask.shape
        left_xs, right_xs, rows = [], [], []
        for y in range(h):
            px = np.where(mask[y] > 0)[0]
            if len(px) > 10:
                left_xs.append(px[0])
                right_xs.append(px[-1])
                rows.append(y)
        if len(rows) < 20:
            return None, None
        rows = np.array(rows)
        return np.column_stack([rows, np.array(left_xs, dtype=float)]), \
               np.column_stack([rows, np.array(right_xs, dtype=float)])
    
    def _detect_anomalies(self, edge_profile, side, img_h, img_w, smooth_window, min_damage_length, edge_bbox_width):
        rows = edge_profile[:, 0].astype(int)
        x_coords = edge_profile[:, 1]
        smooth_x = uniform_filter1d(x_coords, size=smooth_window)
        abs_dev = np.abs(x_coords - smooth_x)
        
        threshold = max(np.mean(abs_dev) + self.deviation_threshold * np.std(abs_dev), 5.0)
        anomaly_mask = abs_dev > threshold
        
        detections = []
        segments = self._find_segments(anomaly_mask, min_damage_length)
        
        for si, ei in segments:
            y_min, y_max = int(rows[si]), int(rows[ei])
            seg_x = x_coords[si:ei+1]
            seg_s = smooth_x[si:ei+1]
            x_min = max(0, int(min(seg_x.min(), seg_s.min())) - edge_bbox_width // 4)
            x_max = min(img_w, int(max(seg_x.max(), seg_s.max())) + edge_bbox_width // 4)
            if (x_max - x_min) < edge_bbox_width // 2:
                cx = (x_min + x_max) // 2
                x_min, x_max = max(0, cx - edge_bbox_width//2), min(img_w, cx + edge_bbox_width//2)
            if (y_max - y_min) < 20:
                cy = (y_min + y_max) // 2
                y_min, y_max = max(0, cy - 20), min(img_h, cy + 20)
            conf = min(1.0, np.max(abs_dev[si:ei+1]) / (threshold * 3))
            detections.append(Detection((x_min, y_min, x_max, y_max), 'edge_damage', conf))
        
        return detections
    
    @staticmethod
    def _find_segments(mask, min_length=10):
        segments, start = [], None
        for i in range(len(mask)):
            if mask[i]:
                if start is None: start = i
            else:
                if start is not None:
                    if (i - start) >= min_length: segments.append((start, i-1))
                    start = None
        if start is not None and (len(mask) - start) >= min_length:
            segments.append((start, len(mask)-1))
        return segments


class BeltDamageDetector:
    """Complete belt damage detection pipeline: YOLOv8 segmentation + CV damage detection."""
    
    def __init__(self, model_path, seg_conf=0.5, scratch_params=None, edge_params=None):
        from ultralytics import YOLO
        self.model = YOLO(model_path)
        self.seg_conf = seg_conf
        self.scratch_detector = ScratchDetector(**(scratch_params or {}))
        self.edge_detector = EdgeDamageDetector(**(edge_params or {}))
    
    def get_belt_mask(self, image):
        results = self.model(image, conf=self.seg_conf, verbose=False)
        if results and results[0].masks is not None:
            best_mask, best_area = None, 0
            for mask in results[0].masks.data:
                mask_np = mask.cpu().numpy()
                area = np.sum(mask_np > 0.5)
                if area > best_area:
                    best_area, best_mask = area, mask_np
            if best_mask is not None:
                h, w = image.shape[:2]
                belt_mask = cv2.resize(best_mask, (w, h), interpolation=cv2.INTER_LINEAR)
                return (belt_mask > 0.5).astype(np.uint8) * 255
        return None
    
    def detect(self, image):
        belt_mask = self.get_belt_mask(image)
        if belt_mask is None:
            return []
        scratches = self.scratch_detector.detect(image, belt_mask)
        edge_damages = self.edge_detector.detect(image, belt_mask)
        return scratches + edge_damages


def draw_detections(image, detections):
    """Draw bounding boxes on image."""
    annotated = image.copy()
    colors = {'scratch': (0, 0, 255), 'edge_damage': (0, 165, 255)}
    for det in detections:
        x1, y1, x2, y2 = det.bbox
        color = colors.get(det.damage_type, (255, 255, 255))
        thickness = max(2, int(min(image.shape[:2]) * 0.003))
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, thickness)
        label = f"{det.damage_type} ({det.confidence:.2f})"
        font_scale = max(0.5, min(image.shape[:2]) * 0.0006)
        font_thickness = max(1, int(font_scale * 2))
        (tw, th), bl = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)
        cv2.rectangle(annotated, (x1, y1 - th - bl - 5), (x1 + tw, y1), color, -1)
        cv2.putText(annotated, label, (x1, y1 - bl - 2), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), font_thickness)
    return annotated


def detections_to_json(detections):
    """Convert detections to JSON format: {\"1\": {\"bbox_coordinates\": [x1,y1,x2,y2]}, ...}"""
    return {str(i): {'bbox_coordinates': list(d.bbox)} for i, d in enumerate(detections, 1)}


print("✅ Damage detection module loaded!")

## 7. Run Inference Pipeline 🔍

In [ ]:
import json
import time
from pathlib import Path

def run_pipeline(image_dir, output_dir, model_path, seg_conf=0.5,
                 scratch_thresh=2.5, edge_thresh=3.0, verbose=False):
    """Run the complete damage detection inference pipeline."""
    image_dir = Path(image_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
    image_files = sorted([f for f in image_dir.iterdir() if f.suffix.lower() in image_extensions])
    
    if not image_files:
        print(f"❌ No images found in {image_dir}")
        return
    
    print(f"Found {len(image_files)} images")
    print(f"Model: {model_path}")
    print(f"Output: {output_dir}\n")
    
    # Initialize detector
    print("Loading model...")
    detector = BeltDamageDetector(
        model_path=model_path,
        seg_conf=seg_conf,
        scratch_params={'thresh_factor': scratch_thresh},
        edge_params={'deviation_threshold': edge_thresh},
    )
    print("Model loaded!\n")
    
    total_det, total_scratch, total_edge = 0, 0, 0
    start = time.time()
    
    for idx, img_path in enumerate(image_files, 1):
        img_name = img_path.stem
        
        if verbose:
            print(f"[{idx}/{len(image_files)}] {img_path.name}", end=' ')
        elif idx % 20 == 0 or idx == len(image_files):
            print(f"  Progress: {idx}/{len(image_files)}")
        
        try:
            image = cv2.imread(str(img_path))
            if image is None:
                print(f"  ⚠️ Cannot read: {img_path}")
                continue
            
            detections = detector.detect(image)
            
            n_s = sum(1 for d in detections if d.damage_type == 'scratch')
            n_e = sum(1 for d in detections if d.damage_type == 'edge_damage')
            total_det += len(detections)
            total_scratch += n_s
            total_edge += n_e
            
            if verbose:
                print(f"→ {len(detections)} detections ({n_s} scratch, {n_e} edge)")
            
            # Save annotated image
            if detections:
                annotated = draw_detections(image, detections)
            else:
                annotated = image
            cv2.imwrite(str(output_dir / f"{img_name}.jpg"), annotated)
            
            # Save JSON
            json_data = detections_to_json(detections)
            with open(output_dir / f"{img_name}.json", 'w') as f:
                json.dump(json_data, f, indent=2)
        
        except Exception as e:
            print(f"  ❌ Error on {img_path.name}: {e}")
            continue
    
    elapsed = time.time() - start
    
    print(f"\n{'='*60}")
    print(f"  ✅ Processing Complete!")
    print(f"{'='*60}")
    print(f"  Images processed:  {len(image_files)}")
    print(f"  Total detections:  {total_det}")
    print(f"    - Scratches:     {total_scratch}")
    print(f"    - Edge damage:   {total_edge}")
    print(f"  Time: {elapsed:.1f}s ({elapsed/len(image_files):.2f}s/image)")
    print(f"  Output: {output_dir.resolve()}")
    print(f"{'='*60}")


print("✅ Pipeline function ready!")

In [ ]:
# ============================================================
# 🚀 RUN THE INFERENCE PIPELINE
# ============================================================

# Path to images to process (use the training images for testing,
# or point to a different folder with test images)
IMAGE_DIR = os.path.join(SOURCE_DIR, 'train', 'images')

run_pipeline(
    image_dir=IMAGE_DIR,
    output_dir=OUTPUT_DIR,
    model_path=BEST_MODEL_PATH,
    seg_conf=0.5,
    scratch_thresh=2.5,
    edge_thresh=3.0,
    verbose=False,
)

## 8. Visualize Sample Results

In [ ]:
import matplotlib.pyplot as plt
import json
from pathlib import Path
import random

output_path = Path(OUTPUT_DIR)
result_images = sorted(output_path.glob('*.jpg'))

# Show up to 6 sample images that have detections
images_with_detections = []
for img_path in result_images:
    json_path = output_path / f"{img_path.stem}.json"
    if json_path.exists():
        with open(json_path) as f:
            data = json.load(f)
        if data:  # has detections
            images_with_detections.append((img_path, data))

print(f"📊 {len(images_with_detections)} / {len(result_images)} images have detections")

# Sample and display
samples = images_with_detections[:6] if len(images_with_detections) <= 6 else random.sample(images_with_detections, 6)

if samples:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for idx, (img_path, det_data) in enumerate(samples):
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(img_rgb)
        axes[idx].set_title(f"{img_path.name}\n{len(det_data)} detections", fontsize=9)
        axes[idx].axis('off')
    
    # Hide unused axes
    for idx in range(len(samples), 6):
        axes[idx].axis('off')
    
    plt.suptitle('Sample Detection Results', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No images with detections found.")

## 9. Download Results

In [ ]:
# Zip outputs for download
import shutil

zip_path = os.path.join(WORK_DIR, 'detection_outputs')
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f"✅ Results zipped to: {zip_path}.zip")
print(f"   Size: {os.path.getsize(zip_path + '.zip') / 1e6:.1f} MB")

# Also zip the model weights
weights_zip = os.path.join(WORK_DIR, 'model_weights_export')
shutil.make_archive(weights_zip, 'zip', WEIGHTS_DIR)
print(f"✅ Weights zipped to: {weights_zip}.zip")
print(f"   Size: {os.path.getsize(weights_zip + '.zip') / 1e6:.1f} MB")

# Download (Colab only)
if IN_COLAB:
    from google.colab import files
    print("\n📥 Downloading results...")
    files.download(zip_path + '.zip')
    files.download(weights_zip + '.zip')

if IN_KAGGLE:
    print(f"\n📥 Download from Kaggle Output tab: {zip_path}.zip")
    print(f"📥 Download from Kaggle Output tab: {weights_zip}.zip")

# Save to Drive (Colab)
if IN_COLAB:
    drive_output = '/content/drive/MyDrive/conveyor_belt_results/'
    os.makedirs(drive_output, exist_ok=True)
    shutil.copy2(zip_path + '.zip', drive_output)
    print(f"\n✅ Results also saved to Drive: {drive_output}")

## 10. (Optional) Run on Custom Test Images

Upload test images to a folder and run the pipeline on them.

In [ ]:
# ============================================================
# To run on custom test images:
# 1. Upload images to a folder (e.g., /content/test_images/)
# 2. Update the path below and run this cell
# ============================================================

# CUSTOM_IMAGE_DIR = '/content/test_images'  # ← update this
# CUSTOM_OUTPUT_DIR = os.path.join(WORK_DIR, 'custom_outputs')

# run_pipeline(
#     image_dir=CUSTOM_IMAGE_DIR,
#     output_dir=CUSTOM_OUTPUT_DIR,
#     model_path=BEST_MODEL_PATH,
#     seg_conf=0.5,
#     scratch_thresh=2.5,
#     edge_thresh=3.0,
#     verbose=True,
# )